In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import seaborn as sns

In [2]:
train_data = pd.read_parquet('/Users/hari/Downloads/ch-26-srm/train.parquet')
test_data = pd.read_parquet('/Users/hari/Downloads/ch-26-srm/test.parquet')

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [4]:
X_train = train_data.drop(columns = ['target','X2','Date'])
y_train = train_data['target']
X_test = test_data.drop(columns = ['ID','X2','Date'])

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
class NeuralNetwork(nn.Module):
    def __init__ (self,input_size,hidden_size,num_classes):
        super(NeuralNetwork,self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.LeakyReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.LeakyReLU(),

            nn.Linear(hidden_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_size,num_classes)
        )
        
    def forward(self,x):
        return self.layers(x)
            

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)



In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)

In [10]:
X_train = torch.tensor(X_train,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)

ValueError: could not determine the shape of object type 'DataFrame'

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)  
y_val = le.transform(y_val)

In [12]:
import torch

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [13]:
y_train = torch.tensor(np.array(y_train),dtype=torch.long)
y_val = torch.tensor(np.array(y_val),dtype=torch.long)

/var/folders/vk/1wl47y5s0y11p511c75cq_2r0000gn/T/ipykernel_12407/4033621589.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train),dtype=torch.long)
/var/folders/vk/1wl47y5s0y11p511c75cq_2r0000gn/T/ipykernel_12407/4033621589.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_val = torch.tensor(np.array(y_val),dtype=torch.long)


In [14]:
input_size = X_train.shape[1]
hidden_size = 256
num_classes = len(np.unique(y_train))

In [15]:
model = NeuralNetwork(input_size,hidden_size,num_classes)
optimizer = torch.optim.AdamW(model.parameters(),lr=0.01)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: 0.9 ** (epoch // 1))

In [ ]:
num_epochs = 1
for epoch in range (num_epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs,y_train)
    loss.backward()
    optimizer.step()
    
    scheduler.step()
    
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    test_outputs = model(X_val)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_val).sum().item() / y_val.size(0)
    print(f"Test Accuracy: {accuracy * 100:.2f}%")